<a href="https://colab.research.google.com/github/giorgioganemfederle-dotcom/Cuadra-Zona-Centro/blob/main/Cuadra_Centro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy requests folium ortools

In [ ]:
import pandas as pd
import numpy as np
import requests
import folium

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

print("Programa iniciado correctamente")

Programa iniciado correctamente


In [ ]:
import pandas as pd

datos = [

    # Centro de distribución
    ["León", "Guanajuato", "Centro", 21.1220, -101.6840],

    # Guanajuato
    ["León", "Guanajuato", "Centro", 21.1220, -101.6840],
    ["Irapuato", "Guanajuato", "Centro", 20.6741, -101.3475],
    ["San Miguel de Allende", "Guanajuato", "Centro", 20.9153, -100.7438],

    # Jalisco
    ["Zapopan", "Jalisco", "Centro", 20.7208, -103.3911],
    ["Guadalajara", "Jalisco", "Centro", 20.6597, -103.3496],
    ["San Agustín", "Jalisco", "Centro", 20.5440, -103.4650],
    ["Tlaquepaque", "Jalisco", "Centro", 20.6397, -103.3120],

    #Aguascalientes
    ["Aguascalientes", "Aguascalientes", "Centro", 21.8853, -102.2915],

    # Querétaro
    ["Querétaro", "Querétaro", "Centro", 20.5888, -100.3899],


]

df = pd.DataFrame(
    datos,
    columns=[
        "Ciudad",
        "Estado",
        "Region",
        "Latitud",
        "Longitud"
    ]
)

df

,Ciudad,Estado,Region,Latitud,Longitud
0,León,Guanajuato,Centro,21.1220,-101.6840
1,León,Guanajuato,Centro,21.1220,-101.6840
2,Irapuato,Guanajuato,Centro,20.6741,-101.3475
3,San Miguel de Allende,Guanajuato,Centro,20.9153,-100.7438
4,Zapopan,Jalisco,Centro,20.7208,-103.3911
5,Guadalajara,Jalisco,Centro,20.6597,-103.3496
6,San Agustín,Jalisco,Centro,20.5440,-103.4650
7,Tlaquepaque,Jalisco,Centro,20.6397,-103.3120
8,Aguascalientes,Aguascalientes,Centro,21.8853,-102.2915
9,Querétaro,Querétaro,Centro,20.5888,-100.3899


In [ ]:
print("Número de puntos:", len(df))
print("Centro de distribución: León")
print("Destinos:", len(df) - 1)

Número de puntos: 10
Centro de distribución: León
Destinos: 9


In [ ]:
import folium

mapa = folium.Map(
    location=[24, -105],
    zoom_start=5
)

for _, fila in df.iterrows():

    if fila["Ciudad"] == "León":
        color = "red"
    else:
        color = "blue"

    folium.Marker(
        location=[
            fila["Latitud"],
            fila["Longitud"]
        ],
        popup=(
            f"{fila['Ciudad']}<br>"
            f"{fila['Estado']}<br>"
            f"{fila['Region']}"
        ),
        tooltip=fila["Ciudad"],
        icon=folium.Icon(
            color=color
        )
    ).add_to(mapa)

mapa

In [ ]:
import requests

def obtener_ruta(origen, destino):

    lon1 = origen[1]
    lat1 = origen[0]

    lon2 = destino[1]
    lat2 = destino[0]

    url = (
        f"https://router.project-osrm.org/"
        f"route/v1/driving/"
        f"{lon1},{lat1};{lon2},{lat2}"
        f"?overview=false"
    )

    respuesta = requests.get(url)

    datos_ruta = respuesta.json()

    if datos_ruta["code"] == "Ok":

        ruta = datos_ruta["routes"][0]

        distancia = ruta["distance"] / 1000
        tiempo = ruta["duration"] / 60

        return distancia, tiempo

    else:

        return None, None

In [ ]:
import numpy as np

n = len(df)

matriz_distancia = np.zeros((n, n))
matriz_tiempo = np.zeros((n, n))

for i in range(n):

    for j in range(n):

        if i == j:
            continue

        origen = (
            df.iloc[i]["Latitud"],
            df.iloc[i]["Longitud"]
        )

        destino = (
            df.iloc[j]["Latitud"],
            df.iloc[j]["Longitud"]
        )

        distancia, tiempo = obtener_ruta(
            origen,
            destino
        )

        matriz_distancia[i][j] = distancia
        matriz_tiempo[i][j] = tiempo

print("Matrices creadas.")

In [ ]:
def optimizar_ruta(matriz_costo):

    n = len(matriz_costo)

    # León = índice 0
    manager = pywrapcp.RoutingIndexManager(
        n,
        1,
        0
    )

    routing = pywrapcp.RoutingModel(
        manager
    )

    def costo_callback(
        from_index,
        to_index
    ):

        origen = manager.IndexToNode(
            from_index
        )

        destino = manager.IndexToNode(
            to_index
        )

        return int(
            matriz_costo[origen][destino] * 1000000
        )

    callback_index = (
        routing.RegisterTransitCallback(
            costo_callback
        )
    )

    routing.SetArcCostEvaluatorOfAllVehicles(
        callback_index
    )

    parametros = (
        pywrapcp.DefaultRoutingSearchParameters()
    )

    parametros.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy
        .PATH_CHEAPEST_ARC
    )

    parametros.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic
        .GUIDED_LOCAL_SEARCH
    )

    parametros.time_limit.seconds = 60

    solucion = routing.SolveWithParameters(
        parametros
    )

    if solucion is None:

        return None

    ruta = []

    index = routing.Start(0)

    while not routing.IsEnd(index):

        nodo = manager.IndexToNode(index)

        ruta.append(nodo)

        index = solucion.Value(
            routing.NextVar(index)
        )

    ruta.append(
        manager.IndexToNode(index)
    )

    return ruta

In [ ]:
!pip install -q ortools

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

ruta_optima = optimizar_ruta(
    matriz_costo
)

print("Ruta óptima (índices):", ruta_optima)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 24.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
print("RUTA OPTIMIZADA")
print("================")

for i, nodo in enumerate(ruta_optima):

    print(
        f"{i+1}. "
        f"{df.iloc[nodo]['Ciudad']}"
    )

RUTA OPTIMIZADA
1. León
2. León
3. San Miguel de Allende
4. Querétaro
5. Irapuato
6. Tlaquepaque
7. Guadalajara
8. Zapopan
9. San Agustín
10. Aguascalientes
11. León


In [ ]:
distancia_total = 0
tiempo_total = 0

for i in range(len(ruta_optima) - 1):

    origen = ruta_optima[i]
    destino = ruta_optima[i + 1]

    distancia_total += (
        matriz_distancia[origen][destino]
    )

    tiempo_total += (
        matriz_tiempo[origen][destino]
    )

print(
    f"Distancia total: "
    f"{distancia_total:,.2f} km"
)

print(
    f"Tiempo total: "
    f"{tiempo_total/60:,.2f} horas"
)

Distancia total: 938.41 km
Tiempo total: 13.02 horas


In [ ]:
mapa_ruta = folium.Map(
    location=[24, -105],
    zoom_start=5
)

for numero, nodo in enumerate(ruta_optima):

    fila = df.iloc[nodo]

    if nodo == 0:

        color = "red"

    else:

        color = "blue"

    folium.Marker(
        location=[
            fila["Latitud"],
            fila["Longitud"]
        ],
        popup=(
            f"Parada {numero + 1}<br>"
            f"{fila['Ciudad']}<br>"
            f"{fila['Estado']}"
        ),
        tooltip=(
            f"{numero + 1}. "
            f"{fila['Ciudad']}"
        ),
        icon=folium.Icon(
            color=color
        )
    ).add_to(mapa_ruta)

mapa_ruta

In [ ]:
def obtener_geometria_ruta(ruta):

    puntos = []

    for nodo in ruta:

        fila = df.iloc[nodo]

        puntos.append(
            f"{fila['Longitud']},"
            f"{fila['Latitud']}"
        )

    coordenadas = ";".join(puntos)

    url = (
        "https://router.project-osrm.org/"
        "route/v1/driving/"
        f"{coordenadas}"
        "?overview=full"
        "&geometries=geojson"
    )

    respuesta = requests.get(url)

    return respuesta.json()

In [ ]:
geometria = (
    datos_gps["routes"][0]["geometry"]
)

folium.GeoJson(
    geometria,
    name="Ruta optimizada Cuadra"
).add_to(mapa_ruta)

folium.LayerControl().add_to(
    mapa_ruta
)

mapa_ruta